In [0]:
%run /Workspace/Users/gustavosousa.md20@gmail.com/retreino_desloc_databricks/00_struct_table


### Treinamento

In [0]:
from sklearn.model_selection import train_test_split

df = spark.table("tbl_ml").toPandas()
X = df.drop(columns=["vlr_pago"])
Y = df["vlr_pago"]

X_train, X_test, y_train, y_test = train_test_split(
    X, Y,
    test_size=0.2,
    random_state=42
)

In [0]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsRegressor

# Desabilitar o MLFLOW automatico do Databricks
import mlflow
mlflow.autolog(disable=True)

colunas_minmax = [
    'dia', 'dia_semana', 'mes', 'hora', 'distancia','latitude_origem',
    'longitude_origem', 'latitude_destino', 'longitude_destino'
]

preprocessador = ColumnTransformer(
    transformers=[
        ('drop', 'drop', ['periodo']),
        ('minmax', MinMaxScaler(), colunas_minmax)
    ],
    remainder='passthrough'  # mantém as outras colunas sem alteração
)

pipeline = Pipeline(steps=[
    ('preprocessamento', preprocessador),
    ('knn', KNeighborsRegressor(n_neighbors=5))
])

pipeline.fit(X_train, y_train)

In [0]:
y_hat_train = pipeline.predict(X_train)
y_hat_test = pipeline.predict(X_test)


### Métricas

In [0]:
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, root_mean_squared_error

metrics = dict(
    mae_train = mean_absolute_error(y_train, y_hat_train),
    mae_test = mean_absolute_error(y_test, y_hat_test),
    rmse_train = root_mean_squared_error(y_train, y_hat_train),
    rmse_test = root_mean_squared_error(y_test, y_hat_test)
)

pd.DataFrame(metrics, index=[0])

In [0]:
e_train = y_train - y_hat_train
e_test = y_test - y_hat_test

df_plot = pd.DataFrame({
    'e_train': e_train,
    'e_test': e_test,
    'y_train': y_train,
    'y_test': y_test,
})

df_plot[["e_train", "e_test"]].hist(bins=20, figsize=(10, 5));

In [0]:
df_plot[["e_train", "y_train"]].plot(
    kind="scatter", x="y_train", y="e_train",
    grid=True, title="Train X Erro"
);

In [0]:
df_plot[["e_test", "y_test"]].plot(
    kind="scatter", x="y_test", y="e_test",
    grid=True, title="Train X Erro"
);

In [0]:
def print_metrics(y_train, y_test, y_hat_train, y_hat_test):
    metrics = dict(
        mae_train = mean_absolute_error(y_train, y_hat_train),
        mae_test = mean_absolute_error(y_test, y_hat_test),
        rmse_train = root_mean_squared_error(y_train, y_hat_train),
        rmse_test = root_mean_squared_error(y_test, y_hat_test)
    )

    return pd.DataFrame(metrics, index=[0])

In [0]:
dfs = []
for i in range(0, 100):
    X_train, X_test, y_train, y_test = train_test_split(
        X, Y,
        test_size=0.2,
    )

    pipeline.fit(X_train, y_train)

    y_hat_train = pipeline.predict(X_train)
    y_hat_test = pipeline.predict(X_test)
    dfs.append(print_metrics(y_train, y_test, y_hat_train, y_hat_test))

In [0]:
pd.concat(dfs).describe()


### Testando o Algoritmo

O algoritmo em média tem um erro de R$ 1,5 e um erro penalizado de R$ 2,0 para valores de corridas muito altos ou muito baratos.

In [0]:
m = pd.DataFrame(metrics, index=[0])
m.columns = ["Erro em Treino", "Erro em Teste", "Erro Penalizado em Treino", "Erro Penalizado em Teste"]
m

In [0]:
print(f"Valor total arrecadao pelas corridas nos dados reais em teste: R$ {y_hat_test.sum():.2f}")
print(f"Valor total arrecadado pelas corridas com o algoritmo em teste: R$ {y_test.sum():.2f}")